# 07d · 结构特征和图：AlphaFold

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计 ✅   07b 新数据 ✅   07c 标签 ✅
              ├─ 07d 结构特征和图：AlphaFold   ← 你在这里
              ├─ 07e 语言模型特征：ESM-2
              └─ 07f 特征表 + 第一个基线模型 + MLflow
```

## 背景：为什么需要这一章

07c 之后，457 个蛋白的突变都有了标签，也就是"考题的答案"。现在开始做**特征**，也就是 07a 说的"体检报告"。第一类是**结构**：每个氨基酸在蛋白的三维结构里处在什么位置。

**为什么结构重要**：埋在蛋白内部的氨基酸，周围挤满了邻居，换一个大小或电荷不同的氨基酸，往往会把整个结构撑坏；露在表面的，换了常常没事。旧 QAFI 的 pLDDT、colasi、接触能量，都是在描述这件事。

结构从哪里来：**AlphaFold** 给几乎每个人类蛋白都预测了三维结构，03b 号已经把它搬进了 raw。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 5 步：在本机比一比旧写法和新写法的速度
   └─ 第 6 步：Databricks 的新流水线 structures：AlphaFold 进 bronze，再算出每个残基的特征
看结果：这里的输出；Databricks 网页的 Jobs 和 Catalog → qafi_dev → silver
```

## 这一章做的 7 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 结构特征是什么 | 住在楼里哪一层、有几个邻居 |
| 2 | 为什么单独一条流水线 | 不同保质期的货，分开进 |
| 3 | 原料：AlphaFold v6 | 先核对图纸和实物对不对得上 |
| 4 | 长蛋白切段 | 拼接照片，取最清楚的那张 |
| 5 | 加速：大计算怎么变快 | 先找最慢的工序，再让每个工人都有活干 |
| 6 | 跑起来：增量计算 | 只做还没做的 |
| 7 | 看结果 | TP53 的热点突变 |

**花费：0**。**时间**：大约 15 分钟。

---

## 准备

运行下面这一格：连上 Databricks 和 AWS。**不用看代码。**

In [ ]:
import gzip, os, subprocess, sys
from pathlib import Path
import boto3
from databricks.sdk import WorkspaceClient
from qafi_platform import ingest, lakehouse
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "databricks" / "src"))   # the code the Databricks job runs
from lakehouse import structure
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
CATALOG, RAW = "qafi_dev", "qafi-186972156153-dev-raw"
print("Databricks:", w.config.host)

---

## 第 1 步 · 结构特征是什么

### 在干什么
弄清楚每个残基要算哪些结构特征，各说明什么。

### 背景
比喻：**一个人住在楼里的处境**：住在中间层还是顶楼，门口有几个邻居，房子是不是危房。

| 特征 | 在说什么 | 比喻 |
|---|---|---|
| **pLDDT** | AlphaFold 对这个位置的预测有多有把握，0–100。低于 50 的地方常常是没有固定结构的"柔性区" | 图纸上这一块画得清不清楚 |
| **RSA**（相对溶剂可及性） | 这个残基露在表面的面积，占它完全暴露时的比例。0 = 完全埋在里面，约 1 = 完全露在外面 | 窗户朝外还是朝里 |
| **contacts_8、contacts_12** | 8 Å、12 Å 范围内有几个残基 | 门口和整层楼各住了几户 |
| **HSE**（半球暴露度） | 侧链那一侧有几个邻居、背面有几个 | 前门邻居多，还是后门邻居多 |
| **邻居列表** | 10 Å 内所有残基和距离 | 通讯录：这就是 08 号图神经网络的"图" |

### 用了什么技术，为什么
- **Biopython**：生物信息学最常用的 Python 库，用来读结构文件
- **表面积**用 **Shrake-Rupley** 方法（拿一个水分子大小的小球在蛋白表面滚一圈），**半球暴露度**用 Hamelryck 2005 的方法。算法和 Biopython 里的一模一样，但改写成了矩阵运算，快很多（第 5 步讲为什么、怎么做）
- **RSA 的分母**：每种氨基酸完全暴露时的面积，用的是 Tien 2013 的理论值，这是现在最常用的一套
- 距离用 **Cβ 原子**（甘氨酸没有 Cβ，用 Cα）：Cβ 指向侧链的方向，比 Cα 更能代表"这个氨基酸占的位置"

### 公司里什么情况怎么做
结构特征是蛋白工程、药物设计里最常用的特征之一，也是 ThermoMPNN、AlphaMissense 这类模型的输入。**先用标准工具算标准特征**，再考虑让深度学习自己学（08 号）。

### 面试怎么说
> "From each AlphaFold model I compute per-residue confidence, relative solvent accessibility, contact counts, half-sphere exposure and a 10 Å neighbour list, which doubles as the residue graph for the GNN."

---

## 第 2 步 · 为什么单独一条流水线

### 在干什么
结构特征放在一条新的 Databricks 流水线 `structures` 里，不放进每月跑的 `training_data`。

### 背景
比喻：**不同保质期的货，分开进**。ClinVar 每个月出新版，所以 `training_data` 每月跑；AlphaFold 一两年才出一个新版本，结构没变就不用重算，而算一次全蛋白组要约 35 分钟。放在一起，每个月都要白白重算一遍。

```
structures（AlphaFold 换版本时才跑）：
copy_raw ──▶ bronze_alphafold ──▶ silver_structure
```

### 公司里什么情况怎么做
**按更新频率拆分流水线**是很常见的设计：每天来的数据、每月来的数据、几年才变的参考数据，各自一条，互不拖累。它们之间的依赖要写清楚：`silver_structure` 要用到 `training_data` 里的 UniProt 序列，图纸里有注释。

### 面试怎么说
> "Pipelines are split by refresh cadence: monthly data in one job, AlphaFold structures, which change only with a new release, in another."

---

## 第 3 步 · 原料：AlphaFold v6

### 在干什么
确认用哪个版本的 AlphaFold，它和我们的 UniProt 序列能不能对上。

### 背景
raw 里有 v4（旧论文用的）和 v6（2025 年 8 月发布，最新）。AlphaFold 给每个蛋白存了两种格式（mmCIF 和 PDB），内容一样，我们保留 PDB。

**最要紧的检查**：特征是按"UniProt 第几位"拼到突变上的。如果 AlphaFold 用的序列和 UniProt 2026_03 的不一样（UniProt 后来改过序列），位置就会错开。我抽查了约 1,400 个蛋白：**v6 和 UniProt 有 99.2% 完全一样**。不一样的蛋白不用，状态里写明原因。

### 用了什么技术，为什么
- 用 **v6**：新模型用最新的结构
- **bronze 原样保存**：每个模型一行，PDB 文件**压缩着**存进去（`pdb_gz`）。2 万多个文件，只占大约 0.6 GB

看一个真实的 AlphaFold 文件长什么样（测试里用的那个 64 个氨基酸的小蛋白）：

In [ ]:
text = gzip.decompress((ROOT / "tests/unit/data/AF-A0A024R1R8-F1-model_v6.pdb.gz").read_bytes()).decode()
print("\n".join(line for line in text.splitlines() if line.startswith("ATOM"))[:400])
structure.fragment_features(text)[["position", "aa", "plddt", "rsa", "contacts_8", "contacts_12", "hse_up", "hse_down"]].head(8)

### 结果怎么看
- 每行 `ATOM` 是一个原子：原子名、氨基酸、第几位、x/y/z 坐标；倒数第二列就是 **pLDDT**（这个蛋白开头是 57.69，把握不大）
- 下面是算出来的特征：每个残基一行

### 面试怎么说
> "Features are joined to variants by UniProt position, so I only use AlphaFold models whose sequence equals the current UniProt sequence; 99% of v6 models do."

---

## 第 4 步 · 长蛋白切段

### 在干什么
处理 AlphaFold 切成好几段的长蛋白。

### 背景
超过 2,700 个氨基酸的蛋白，AlphaFold 分段预测：每段 1,400 个残基，每往后挪 200 个残基切一段，所以相邻两段大面积重叠（F1、F2、F3……）。全蛋白组有 216 个这样的蛋白，我们有标签的 457 个里有 16 个，比如 OBSCN、NEB，都是 Domainome 测过结构域的超长蛋白。最长的是 **TTN（肌联蛋白）**：34,350 个氨基酸，**166 段**。

比喻：**拼接全景照片**。同一个地方在两张照片里都有，取它在哪张照片里**最靠中间**的那张，因为照片边缘最容易变形。AlphaFold 的片段也一样，两头的预测最不可靠。

### 用了什么技术，为什么
- **每段对到 UniProt 的哪个位置**：拿这段的序列去 UniProt 序列里找，找到唯一的位置就对上了。不靠"第几段从第几位开始"的约定，约定变了也不会错
- **每个残基选最居中的那一段**：取它离这段两头距离较小的那个值，选这个值最大的那一段
- **先定位，再计算，每段只留自己负责的残基**：先用很便宜的办法（只读每段的 Cα 行，拿到序列）把所有段定好位、分好工；再一段一段算特征，每段算完只留下归它管的残基。这样内存只和蛋白的长度有关，和段数无关。第一版是把每段的结果全留着、最后再去重，TTN 166 段 × 1,400 个残基，一个蛋白就占 1 GB 内存，在 Databricks 上把 Python 进程撑爆了（第 5 步细讲）

单元测试里专门造了两段重叠的片段，检查拼接结果。

### 面试怎么说
> "Proteins over 2,700 residues come as overlapping AlphaFold fragments. I place each fragment by finding its sequence in UniProt and take every residue from the fragment where it sits furthest from the ends."

---

## 第 5 步 · 加速：大计算怎么变快

### 在干什么
让全蛋白组 2 万个蛋白的结构特征算得快，而且结果一点不变。

### 背景
第一次把全蛋白组交给 Databricks：跑满 2 小时超时，**一个结果也没存下来**。再拿 2,000 个蛋白试：31 分钟，照这个速度全部要 5 个多小时。

比喻：**工厂的流水线太慢**。先别急着加工人，先看两件事：哪道工序最慢？是不是有工人闲着？

加速以后又出了第二个问题：跑到 14,300 个蛋白时任务失败，报错 `[UDF_PYSPARK_ERROR.OOM] Python worker exited unexpectedly (crashed) due to running out of memory`：**内存不够**。那一批里有 TTN（166 段）。在本机复现：TTN 一个蛋白占 1 GB。原因是第一版把 166 段的所有残基先全部留着，最后再去重。改成第 4 步说的"先定位、每段只留自己负责的"，**降到 465 MB，结果逐行一样**。

公司里的标准顺序：

| 顺序 | 做什么 | 我们这里 |
|---|---|---|
| 1 | **先量**：找出最慢的那一步。凭感觉猜，常常猜错 | 78% 的时间花在半球暴露度上 |
| 2 | **换写法**：把最慢的那步改快，比加机器便宜得多 | 改成矩阵运算，结果和原来一模一样 |
| 3 | **让每个核都有活干**：切成很多小任务 | 每 20 个蛋白一个任务 |
| 4 | **分批保存**：长任务中途断了不白干 | 每 2,000 个蛋白存一次（第 6 步）；这次 OOM 之前的 14,300 个都没白算 |
| 5 | **最后才加机器** | 免费版算力有上限，所以前面几步更重要 |

### 用了什么技术，为什么
- **先量：`cProfile`**：Python 自带的计时工具，列出每个函数花了多少时间。量出来：78% 在半球暴露度，其次是表面积
- **换写法：向量化（vectorization）**：Biopython 算半球暴露度时，用 Python 一对一对残基地循环，一个 500 个残基的蛋白要循环 25 万次，Python 循环很慢。改成 **numpy 一次算整张矩阵**，循环交给底层的 C 语言。表面积也一样：原来一个原子一个原子地查，改成用 **scipy 的 k-d 树**（按空间位置建的索引，像地图上的网格）一次查完
- **怎么保证结果不变**：单元测试拿同一个结构，分别用 Biopython 和我们的写法算，**逐个残基核对，完全一样**。优化性能的铁律：先有测试，再改代码
- **切小任务：`repartition`**：这份数据**字节少、计算重**：2,000 个蛋白的文件才几十 MB。Spark 按字节估算工作量（AQE，自适应查询执行），会把它们合成很少几个大任务，大部分核闲着。`repartition` 按蛋白切成每 20 个一份的小任务，哪个核空了就领下一个：

```
不切：  核1 [████████████ 1000 个蛋白]   核2 [████████████ 1000 个]   核3 闲   核4 闲 …
切小：  100 个小任务（每个 20 个蛋白），所有核轮流领活
```

### 执行
在本机比一比：下载 TP53 的公开 AlphaFold 结构，分别给 Biopython 最慢的两步计时，再给我们的全部特征计时：

In [ ]:
import io, time, urllib.request
from Bio.PDB import HSExposureCB, PDBParser
from Bio.PDB.SASA import ShrakeRupley
tp53 = urllib.request.urlopen("https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb").read().decode()
def seconds(step):
    start = time.perf_counter()
    step()
    return round(time.perf_counter() - start, 2)
model_1, model_2 = (PDBParser(QUIET=True).get_structure("tp53", io.StringIO(tp53))[0] for _ in range(2))
structure.fragment_features(tp53)   # warm-up: the first call also loads the libraries
print("Biopython, surface area:", seconds(lambda: ShrakeRupley().compute(model_1, level="R")), "s")
print("Biopython, half-sphere :", seconds(lambda: HSExposureCB(model_2, radius=13.0)), "s")
print("ours, every feature    :", seconds(lambda: structure.fragment_features(tp53)), "s")

### 结果怎么看
- Biopython 光是最慢的这两步就要约 0.5 秒；我们的写法把**所有**特征算完，只要 0.1 秒左右（每台电脑、每次运行的数字都不一样，比例差不多）
- 16 个常见蛋白上平均：每个蛋白从 1.39 秒降到 0.20 秒，**快了约 7 倍**
- 在 Databricks 上：2,000 个蛋白从 31 分钟降到约 3.5 分钟，全蛋白组约 35 分钟算完

### 公司里什么情况怎么做
- **慢了先量，不要猜**；优化以后，用测试证明结果没变
- **内存按最大的那一条记录来设计，不按平均的**：平均的蛋白很小，但只要 TTN 一条记录撑爆内存，整批就失败。先在本机拿最大的那条复现、量内存，修好再上云
- Spark 里：**数据大**，交给 AQE 自动分；**数据小但计算重**（每条记录要跑一段复杂计算或一个模型），手动 `repartition` 切小
- 所有核都用满了还不够快，才**加机器**：集群自动扩容，或者用 Lakeflow Jobs 的 **for each** 任务把工作分成几份同时跑
- GPU 只对矩阵运算这类工作有用（比如 08 号的深度学习），这里的结构计算用不上

### 面试怎么说
> "The first proteome-wide run timed out. I profiled it: 78% of the time was half-sphere exposure in a Python double loop. Rewriting it and the surface-area step as vectorised numpy and k-d tree code, with unit tests showing results identical to Biopython, made it seven times faster. I also repartitioned the small-in-bytes, compute-heavy data into small tasks so every core stayed busy."

> "Then a Python worker ran out of memory on titin, which comes as 166 overlapping AlphaFold fragments: the code kept every fragment's residues before de-duplicating. Placing the fragments first and keeping only the residues each one supplies halved peak memory with identical output."

---

## 第 6 步 · 跑起来：增量计算

### 在干什么
用新流水线算出全蛋白组每个残基的特征。

### 背景
全蛋白组 2 万个蛋白，加速以后也要约 35 分钟。长任务最怕中途断了要从头来。

比喻：**只做还没做的**。每算完一批（2,000 个蛋白），就存起来、记下这批"已完成"，日志里打一行进度；下次运行只算没完成的。所以：
- 中途断了，重跑会**接着算**
- 全部算完以后再跑，会显示 **0 个新蛋白**，几乎不花时间
- AlphaFold 换新版本时，用 `rebuild=true` 清空重来

### 用了什么技术，为什么
- **增量处理**：先查 `silver.alphafold_proteins` 里哪些蛋白已经有结果，只把剩下的交给 Spark
- **分批保存**：每批算完立刻写进两张 silver 表，中途断了最多损失当前这一批（`batch_size` 参数可以调）
- **applyInPandas**：和 07c 一样，每个蛋白交给一个测试过的 Python 函数，Spark 分给多台机器同时算
- **Biopython 在 Databricks 上**：任务的第一格用 `%pip install biopython==1.88` 装进这次运行的环境，版本写死
- **一个蛋白出错不影响别的**：读不了的模型记下原因（`status`），其他照常算

### 执行
先放一份提货单（AlphaFold v6 的包，5.2 GB；04 号讲过：免费版 Databricks 读不到我们的 S3）：

In [ ]:
keys = [s["key"] for s in ingest.load_sources(ROOT / "config/sources.toml") if s["dataset"] == "alphafold-human" and s["version"] == "v6"]
lakehouse.upload_transfer_list(w, CATALOG, lakehouse.transfer_links(aws.client("s3"), RAW, keys, hours=6))

部署，然后跑整条 `structures` 流水线：

In [ ]:
run("cd databricks && databricks bundle deploy")
run("cd databricks && databricks bundle run structures")
lakehouse.latest_run(w, "qafi structures")

### 结果怎么看
- `copy_raw`：文件已经在 Databricks 里，大小对，跳过
- `bronze_alphafold`：大约 5 分钟，重新读一遍 5 GB 的包
- `silver_structure`：输出里写着 **0 proteins to compute**：我写这一章时已经把全蛋白组算完了，增量设计让它不重复算

### 去网页上看
Databricks → **Jobs & Pipelines** → `qafi structures`：3 个方框；Catalog → `silver` → `residue_structure`、`alphafold_proteins`。

### 面试怎么说
> "The structure step is incremental: it computes only proteins without results, so an interrupted run resumes and a repeated run costs nothing; a rebuild flag starts over for a new AlphaFold release."

---

## 第 7 步 · 看结果

### 在干什么
看全蛋白组算出了多少，再看一个熟悉的例子。

In [ ]:
sql(w, f"SELECT status, count(*) AS proteins, sum(residues) AS residues, round(avg(mean_plddt), 1) AS mean_plddt FROM {CATALOG}.silver.alphafold_proteins GROUP BY status ORDER BY proteins DESC")

### 结果怎么看
- `ok`：结构特征全部算好的蛋白，**20,467 个，占 99.8%**，一共 1,140 万个残基
- `sequence differs from UniProt`：37 个，AlphaFold 的序列和 UniProt 2026_03 不一样，不用
- `covers part of the sequence`：7 个，模型只覆盖了部分序列

**TP53 的几个癌症热点突变**：R175、R248、R273 是 TP53 里最常见的致癌突变位置。看看它们在结构里的处境：

In [ ]:
sql(w, f'''
SELECT position, aa, plddt, round(rsa, 2) AS rsa, contacts_12, hse_up, hse_down, size(neighbours) AS neighbours_10A
FROM {CATALOG}.silver.residue_structure
WHERE uniprot_id = 'P04637' AND position IN (175, 245, 248, 249, 273, 282)
ORDER BY position''')

### 结果怎么看
- 这几个位置的 **pLDDT 都在 95 以上**：在 DNA 结合结构域里，结构很确定
- **R175、G245、R249、R282** 都埋在里面（RSA 0.03–0.14，邻居多）：突变后蛋白折叠不稳，文献里叫"**结构型**"突变
- **R248** 完全露在外面（RSA 0.72），**R273** 也露出一部分（0.29）：它们本来是**直接接触 DNA** 的，文献里叫"**接触型**"突变。AlphaFold 预测的是没有 DNA 的单个蛋白，所以看起来是暴露的。这提醒我们：**结构特征只描述折叠，描述不了和别的分子的结合**。08 号会再遇到这个问题

有标签的突变里，有多少找得到结构特征：

In [ ]:
sql(w, f'''
SELECT count(*) AS labels, round(avg(CASE WHEN s.uniprot_id IS NOT NULL THEN 1.0 ELSE 0 END), 4) AS with_structure
FROM {CATALOG}.gold.fct_variant_label l
LEFT JOIN {CATALOG}.silver.residue_structure s ON s.uniprot_id = l.uniprot_id AND s.position = l.uniprot_position''')

### 结果怎么看
107 万个有标签的突变，**100%** 都有结构特征：457 个有标签的蛋白，在 AlphaFold v6 里全部是 `ok`。

### 公司里什么情况怎么做
- 算完特征，先看**覆盖率**（多少数据有特征）和**几个你懂的例子**（TP53 的热点）是否合理，再拿去训练
- 发现特征的局限（这里是"看不到结合"），记下来，作为以后改进的方向

### 面试怎么说
> "Sanity checks after feature engineering: coverage of the labelled variants, and known examples. TP53's structural hotspots such as R175 come out buried, while the DNA-contact hotspots R248 and R273 look exposed, because a single-chain AlphaFold model has no DNA."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07d AlphaFold structure features and residue graphs") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 为什么 |
|---|---|---|
| 1 特征 | pLDDT、RSA、接触数、半球暴露度、10 Å 邻居 | 描述每个残基在结构里的处境 |
| 2 流水线 | 单独的 `structures` | 结构几年才变，不该每月重算 |
| 3 原料 | AlphaFold v6，序列和 UniProt 核对 | 位置对不上，特征就挂错地方 |
| 4 长蛋白 | 按序列找位置，取最居中的片段；先定位，每段只留自己负责的 | 片段两头的预测最不可靠；内存只随蛋白长度增长 |
| 5 加速 | 先量；最慢的两步改成矩阵运算；切成小任务 | 全蛋白组从 5 个多小时降到约 35 分钟，结果不变 |
| 6 增量 | 只算没算过的；可续跑 | 长任务不怕中断，重跑不浪费 |
| 7 检查 | 覆盖率 + TP53 热点 | 先确认合理，再训练 |

**一句话**：全蛋白组每个残基的结构特征都算好了，训练和以后预测全蛋白组用的是同一张表；每个蛋白的邻居列表，也为 08 号的图神经网络准备好了。

## 面试题

1. **Why compute features for the whole proteome when you only train on 457 proteins?**
   > "Training and prediction must see features computed by the same code from the same source. Computing once for every protein removes training-serving skew, and the proteome-wide predictions need them anyway."

2. **How do you make a long batch job safe to rerun?**
   > "Make it incremental and idempotent: record what is done and compute only the rest, so a crash resumes and a repeat is free; keep an explicit rebuild switch for when the inputs change."

3. **A Spark job is slow, but its input is small. What do you check?**
   > "Whether the work is spread over enough tasks: adaptive query execution sizes tasks by bytes, so small but compute-heavy data lands in a few tasks while other cores sit idle; repartition it into small tasks. Then profile the per-record code and vectorise the hotspot."

4. **What can't AlphaFold structural features tell you?**
   > "Interactions with other molecules: a single-chain model has no DNA or partner proteins, so interface residues such as TP53 R248 look exposed even though mutating them is deleterious."

**下一步 07e · 语言模型特征**：用 ESM-2 给每个突变打分（它学过几亿条蛋白序列），并存下每个残基的向量，留给 08 号。